In [7]:
import pandas as pd
import numpy as np
import warnings
from astropy.io import fits
from astropy import modeling
from matplotlib import pyplot as plt
from matplotlib.ticker import (MultipleLocator, AutoMinorLocator)
from astropy.constants import c
from astropy.modeling import models
from astropy import units as u
from specutils.spectra import Spectrum1D, SpectralRegion, Spectrum
from specutils.fitting import fit_lines, fit_generic_continuum
from specutils.analysis import equivalent_width
from specutils.fitting import find_lines_derivative
from astropy.stats import sigma_clip
from astropy.table import Table
import os
import sys
from PyAstronomy import pyasl
from PyAstronomy.pyasl import fastRotBroad, rotBroad, specAirVacConvert, airtovac2, equidistantInterpolation
from scipy.interpolate import interp1d
from starfused import StellarModel

In [3]:
path = '/Users/brooklyngustaf/Desktop/research/Brooklyn_NEID_spectra_project/l2fits/'
folder_path = path + 'TIC_168707425'

def get_spectral_order(fits_file, echelle_order):
    all_fluxes = []
    all_wavelengths = []
    # SN_list = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".fits"):
            file_path = os.path.join(folder_path, filename)
            with fits.open(file_path) as hdul:
                header = dict(hdul[12].header['*DVR\D+|*RV\D+|*CCFJDSUM*|ccfrvmod|'].items())
                header.update(dict(hdul[0].header['OBJECT*|*DATE-OBS|*SCI-OBJ|*QRA|*QDEC|OBSJD|*QSPT'].items()))
                header.update({'rv_correction':hdul[0].header[f'SSBRV{173 - echelle_order:03d}']})
                SN = hdul[0].header['EXTSNR']
                activity = Table(hdul[13].data)
                # SN_list.append(SN)
                SN_wavelength=hdul[0].header['EXTSNRWL']
                c_kms = c.to('km/s').value  # Speed of light in km/s
                rv = header['CCFRVMOD'] - header['rv_correction'] #correcting from heliocentric rv
                wave = (hdul[7].data[echelle_order] * u.AA) / (1 + (rv / c_kms)) #doppler shift correction
                flux = hdul[1].data[echelle_order] 
                all_wavelengths.append(wave) 
                all_fluxes.append(flux)
                # print(rv)
                # print(SN)
                # print(SN_wavelength)
                # print(activity)

    # Convert to arrays
    all_fluxes = np.array(all_fluxes)
    all_wavelengths = np.array(all_wavelengths)

    # choose reference grid (e.g., first spectrum)
    wavelengths = all_wavelengths[0]

    resampled_fluxes = []

    for wave, flux in zip(all_wavelengths, all_fluxes):
        interp_flux = np.interp(wavelengths, wave, flux)
        resampled_fluxes.append(interp_flux)

    flux = np.mean(resampled_fluxes, axis=0)



    return wavelengths, flux, header

<>:12: SyntaxWarning: invalid escape sequence '\D'
<>:12: SyntaxWarning: invalid escape sequence '\D'
/var/folders/ch/bcxs_6vj0jj86m8mfdlxqyhw0000gn/T/ipykernel_66818/2789882585.py:12: SyntaxWarning: invalid escape sequence '\D'
  header = dict(hdul[12].header['*DVR\D+|*RV\D+|*CCFJDSUM*|ccfrvmod|'].items())


In [4]:
wavelengths, flux, header = get_spectral_order(folder_path, 80)

# Convert Astropy Quantity to plain Angstrom values if needed
wave = wavelengths.value if hasattr(wavelengths, "value") else wavelengths

# If you don't have uncertainties:
err = np.zeros_like(flux)

data = np.column_stack([wave/10, flux, err])

np.savetxt(
    "TIC_168707425_order80.txt",
    data,
    header="waveobs flux err",
    comments="",
    fmt="%.10f"
)

In [18]:
ispec_dir = "/Users/brooklyngustaf/Desktop/research/iSpec_v20230804"
sys.path.insert(0, os.path.abspath(ispec_dir))

import ispec

In [17]:
spectrum = read_spectrum('/Users/brooklyngustaf/Desktop/research/code/TIC_168707425_order80.txt')
plotting('/Users/brooklyngustaf/Desktop/research/code/TIC_168707425_order80.txt')

TypeError: 'module' object is not callable

In [19]:
def interpolate_atmosphere(code="spectrum"):
    #--- Synthesizing spectrum -----------------------------------------------------
    # Parameters
    teff = 4777.0
    logg = 4.44
    MH = 0.00
    alpha = 0.00

    # Selected model amtosphere, linelist and solar abundances
    #model = ispec_dir + "/input/atmospheres/MARCS/"
    model = ispec_dir + "/input/atmospheres/MARCS.GES/"
    #model = ispec_dir + "/input/atmospheres/MARCS.APOGEE/"
    #model = ispec_dir + "/input/atmospheres/ATLAS9.APOGEE/"
    #model = ispec_dir + "/input/atmospheres/ATLAS9.Castelli/"
    #model = ispec_dir + "/input/atmospheres/ATLAS9.Kurucz/"
    #model = ispec_dir + "/input/atmospheres/ATLAS9.KuruczNOVER/"
    #model = ispec_dir + "/input/atmospheres/ATLAS9.KuruczODFNEW/"
    #model = ispec_dir + "/input/atmospheres/ATLAS9.Kirby/"

    # Load model atmospheres
    modeled_layers_pack = ispec.load_modeled_layers_pack(model)

    # Validate parameters
    if not ispec.valid_atmosphere_target(modeled_layers_pack, {'teff':teff, 'logg':logg, 'MH':MH, 'alpha':alpha}):
        msg = "The specified effective temperature, gravity (log g) and metallicity [M/H] \
                fall out of theatmospheric models."
        print(msg)

    # Prepare atmosphere model
    atmosphere_layers = ispec.interpolate_atmosphere_layers(modeled_layers_pack, {'teff':teff, 'logg':logg, 'MH':MH, 'alpha':alpha}, code=code)
    atmosphere_layers_file = "example_atmosphere.txt"
    atmosphere_layers_file = ispec.write_atmosphere(atmosphere_layers, teff, logg, MH, atmosphere_filename=atmosphere_layers_file, code=code)